<a href="https://colab.research.google.com/github/bsheese/387/blob/main/22_images/22_0_Convolution_and_CNNs/22_0_1_Convolution_by_Hand.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Images: Part 1
## Convolution by Hand

---

## What This Notebook Is About

The networks in unit 21 flattened every Fashion-MNIST image into a row of 784 numbers. That works, but it throws away the most basic fact about an image: which pixels are next to each other. To an `nn.Linear` layer, pixel 29 (just below pixel 1) is no more related to pixel 1 than pixel 700 is. Anything about shape, like an edge, a corner or a collar, has to be pieced together from scratch out of separate pixels, and has to be learned again for every position in the image where it might appear.

**Convolutional** layers are built around the shape of the image instead. They slide a small grid of weights, called a **filter**, across the image and compute a weighted sum at every position. The same filter is used everywhere, so a pattern it detects is found wherever it appears.

This notebook works through convolution by hand on tiny examples, checks that PyTorch's `nn.Conv2d` computes the same thing, and then looks at what a few filters do to a real Fashion-MNIST image. 22_0_2 builds a network out of these layers.

**What you will learn:**
1. How a filter slides over an image, and what a **feature map** is
2. That `nn.Conv2d` computes exactly the hand calculation
3. What edge-detecting and blurring filters do to a real image
4. Channels, padding and stride, and how they set the shape of the output
5. Pooling, which shrinks a feature map
6. Why convolutional layers need far fewer weights than `nn.Linear` layers

In [ ]:
import os
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn

DATA_URL = 'https://raw.githubusercontent.com/bsheese/387/main/data/'

def fetch(name):
    # Download a file from the course's data folder, unless it is already here.
    if not os.path.exists(name):
        urllib.request.urlretrieve(DATA_URL + name, name)
    return name

data = np.load(fetch('fashion10k.npz'))
CLASSES = list(data['classes'])

---

## Section 1: One Filter, by Hand

Here is a tiny 6 × 6 grayscale "image": dark (0) on the left, bright (1) on the right, with the change in the middle. That change from dark to bright is a vertical **edge**.

In [ ]:
image = np.array([[0, 0, 0, 1, 1, 1]] * 6, dtype=float)
print(image)

A **filter** is a small grid of weights, here 3 × 3. This one has −1s in its left column, 0s in the middle and +1s on the right:

In [ ]:
vertical_edge = np.array([[-1, 0, 1],
                          [-1, 0, 1],
                          [-1, 0, 1]], dtype=float)

To apply the filter, place it over a 3 × 3 patch of the image, multiply each weight by the pixel under it, and add up the nine products. That's a weighted sum, the same calculation as one neuron in 19_1_3, except that the inputs are a small patch of neighboring pixels. Then slide the filter one pixel to the right and do it again, and so on across and down the image. Each position gives one number, and the grid of results is called a **feature map**.

Over the top-left patch, all the pixels are 0, so the result is 0. Over a patch that straddles the edge, the right column is brighter than the left, so the result is large:

In [ ]:
top_left = image[0:3, 0:3]
on_edge = image[0:3, 2:5]
print('top-left patch:\n', top_left, '\n  result:', (top_left * vertical_edge).sum())
print('patch on the edge:\n', on_edge, '\n  result:', (on_edge * vertical_edge).sum())

Doing this at every position is a double loop. A 3 × 3 filter fits in 4 positions across and 4 down a 6 × 6 image, so the feature map is 4 × 4.

In [ ]:
def convolve(image, filt):
    k = filt.shape[0]
    rows = image.shape[0] - k + 1
    cols = image.shape[1] - k + 1
    out = np.zeros((rows, cols))
    for i in range(rows):
        for j in range(cols):
            patch = image[i:i + k, j:j + k]
            out[i, j] = (patch * filt).sum()     # one weighted sum per position
    return out

feature_map = convolve(image, vertical_edge)
print(feature_map)

The feature map is 0 wherever the filter sits over flat color and 3 wherever it straddles the edge. The filter is an **edge detector**: it responds where brightness increases from left to right and ignores everything else. An edge in the other direction, from bright to dark, would give −3.

The filter found the edge without being told where to look. The same nine weights were used at every position, so it would find a vertical edge anywhere in the image.

---

## Section 2: PyTorch's `nn.Conv2d`

`nn.Conv2d` is PyTorch's convolutional layer. Its weights start out random, like any layer, but we can copy our filter into it and check that it computes the same feature map. (Strictly, what the layer computes is called cross-correlation; deep learning calls it convolution anyway.)

PyTorch expects images with a batch dimension and a **channel** dimension: shape (batch, channels, height, width). A grayscale image has 1 channel and a color image has 3 (red, green, blue), as in 20_0_1.

In [ ]:
conv = nn.Conv2d(in_channels=1, out_channels=1, kernel_size=3, bias=False)
print('weight shape:', conv.weight.shape)      # (out_channels, in_channels, 3, 3)

with torch.no_grad():
    conv.weight[0, 0] = torch.tensor(vertical_edge)

x = torch.tensor(image, dtype=torch.float32).reshape(1, 1, 6, 6)     # batch of 1, 1 channel
with torch.no_grad():
    print(conv(x)[0, 0])

The same 4 × 4 feature map. In a network, nobody chooses the filter's weights. They start random and are learned by gradient descent, exactly like the weights of the hidden layer that learned XOR in 20_1_1. Edge detectors are among the first things convolutional networks learn on their own, as 22_1_1 shows for ResNet-18.

---

## Section 3: Filters on a Real Image

Here are three hand-made filters applied to a Fashion-MNIST ankle boot:

- **Vertical edges**, the filter from Section 1
- **Horizontal edges**, the same filter turned on its side
- **Blur**, every weight equal to 1/9, which replaces each pixel by the average of its 3 × 3 neighborhood

In [ ]:
filters = {
    'vertical edges': vertical_edge,
    'horizontal edges': vertical_edge.T,
    'blur': np.ones((3, 3)) / 9,
}
boot = data['x_train'][np.where(data['y_train'] == 9)[0][0]] / 255

fig, axes = plt.subplots(1, 4, figsize=(14, 3.6))
axes[0].imshow(boot, cmap='gray')
axes[0].set_title('original (28 x 28)')
for ax, (name, filt) in zip(axes[1:], filters.items()):
    fm = convolve(boot, filt)
    ax.imshow(fm, cmap='RdBu_r' if 'edges' in name else 'gray',
              vmin=-np.abs(fm).max() if 'edges' in name else None, vmax=np.abs(fm).max() if 'edges' in name else None)
    ax.set_title(f'{name} ({fm.shape[0]} x {fm.shape[1]})')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

In the edge maps, red is a strong positive response (getting brighter going right, or going down) and blue a strong negative one (getting darker). Each edge filter lights up along the boot's outline where the outline runs in its direction and stays near 0 on flat areas, inside the boot and in the background. The blur filter keeps the overall shape and smooths away fine detail.

Notice that each feature map is 26 × 26, two pixels smaller than the image in each direction, because a 3 × 3 filter can't be centered on the outermost pixels. Section 4 explains how to keep the size.

A convolutional layer applies many filters at once, each producing its own feature map. With learned filters, a first layer usually ends up with a mix of edge detectors at different angles, blob and spot detectors, and (for color images) color detectors.

---

## Section 4: Channels, Padding and Stride

### Many filters make many channels

A layer with `out_channels=8` has 8 filters and produces 8 feature maps. The output is like an image with 8 channels, and the next convolutional layer treats it that way: each of its filters looks at a 3 × 3 patch of **all** the incoming channels at once. So a filter in the second layer has 3 × 3 × 8 weights and can respond to combinations of the first layer's features, such as a vertical edge next to a horizontal edge, which is a corner.

In [ ]:
x = torch.tensor(boot, dtype=torch.float32).reshape(1, 1, 28, 28)
layer1 = nn.Conv2d(1, 8, kernel_size=3)
layer2 = nn.Conv2d(8, 16, kernel_size=3)
print('input:          ', x.shape)
print('after layer 1:  ', layer1(x).shape, '  weights', layer1.weight.shape)
print('after layer 2:  ', layer2(layer1(x)).shape, '  weights', layer2.weight.shape)

The weight shape is (out_channels, in_channels, height, width): layer 2 has 16 filters, each 8 × 3 × 3. ResNet-18's first layer in 20_1_1 had weights of shape (64, 3, 7, 7): 64 filters, each looking at a 7 × 7 patch of the 3 color channels.

### Padding keeps the size

Each 3 × 3 layer shrinks the image by 2 pixels. **Padding** adds a border of zeros around the input first, so that the filter can be centered on every pixel, including the ones on the edge. With a 3 × 3 filter, `padding=1` keeps the output the same size as the input.

### Stride skips positions

The **stride** is how far the filter moves each step. With `stride=2` it computes every other position in each direction, so the output is about half as wide and half as tall.

In [ ]:
for settings in [dict(), dict(padding=1), dict(padding=1, stride=2)]:
    layer = nn.Conv2d(1, 8, kernel_size=3, **settings)
    print(f'{str(settings):30} output {tuple(layer(x).shape)}')

The output size follows a formula: for an input of size $n$, filter size $k$, padding $p$ and stride $s$, the output is $\lfloor (n + 2p - k)/s \rfloor + 1$ (where $\lfloor \cdot \rfloor$ means round down). For 28, 3, 1 and 2 that gives $\lfloor 27/2 \rfloor + 1 = 14$. You don't need to memorize it; printing the shape after each layer, as above, is the reliable way to find shape errors.

---

## Section 5: Pooling

**Max pooling** shrinks a feature map by keeping only the largest value in each small block. With 2 × 2 blocks, every block of 4 values becomes 1, so the height and width are halved.

In [ ]:
fm = torch.tensor([[1., 3., 0., 2.],
                   [4., 2., 1., 0.],
                   [0., 1., 5., 6.],
                   [2., 0., 7., 1.]]).reshape(1, 1, 4, 4)
print(nn.MaxPool2d(2)(fm)[0, 0])

The top-left block (1, 3, 4, 2) becomes 4, and so on. Pooling does two useful things:

- It **shrinks** the feature maps, so later layers have fewer numbers to process and each of their filters covers a larger part of the original image.
- It makes the network **less sensitive to exactly where** a feature is. If an edge moves by one pixel but stays in the same 2 × 2 block, the pooled value doesn't change.

A typical convolutional network repeats the same pattern a few times: convolution, activation (ReLU), pooling. The image gets smaller and the number of channels grows, so the network goes from many simple features at full resolution to fewer, more complex features covering larger areas.

---

## Section 6: Why Convolution Needs So Few Weights

Compare a convolutional layer with an `nn.Linear` layer that produces the same number of outputs from a 28 × 28 image:

In [ ]:
conv = nn.Conv2d(1, 32, kernel_size=3, padding=1)            # 32 feature maps of 28 x 28
linear = nn.Linear(784, 32 * 28 * 28)                         # the same number of outputs
count = lambda layer: sum(p.numel() for p in layer.parameters())
print(f'Conv2d(1, 32, 3): {count(conv):>12,} weights')
print(f'Linear(784, {32 * 28 * 28}): {count(linear):>12,} weights')

The convolutional layer needs 320 weights (32 filters of 9 weights, plus 32 biases). The linear layer needs over 19 million for the same number of outputs, because every output is connected to every pixel with its own weight. The convolutional layer gets away with so few because of two assumptions about images:

- **Local connections**: each output only looks at a small neighborhood, since nearby pixels are what matter for edges and textures.
- **Shared weights**: the same filter is used at every position, since an edge is an edge wherever it appears.

Fewer weights means less to learn and less room to memorize, which helps with the overfitting from 21_1_1. And because the same filter is used everywhere, a pattern learned in one part of the image is recognized in every other part. 22_0_2 tests that directly by shifting images a couple of pixels.

[CNN Explainer](https://poloclub.github.io/cnn-explainer/) shows all of these steps (filters sliding over an image, channels, ReLU, pooling) on a small trained network, running in your browser.

---

## Summary

- A **filter** is a small grid of weights. **Convolution** slides it across the image and computes a weighted sum of the pixels under it at every position. The results form a **feature map**.
- An edge-detecting filter responds where brightness changes in its direction and gives 0 on flat areas. In a network, filter weights are learned, not chosen by hand.
- `nn.Conv2d(in_channels, out_channels, kernel_size)` applies `out_channels` filters, each spanning all the input channels. Its weights have shape (out_channels, in_channels, k, k), and images have shape (batch, channels, height, width).
- **Padding** adds a border so the output keeps its size; **stride** skips positions and shrinks the output. Printing shapes layer by layer is the way to check them.
- **Max pooling** keeps the largest value in each block, shrinking feature maps and making the network less sensitive to exactly where a feature is.
- **Local connections** and **shared weights** let a convolutional layer use a few hundred weights where a linear layer would need millions.

**Next:** 22_0_2 builds a convolutional network from these layers and trains it on the same Fashion-MNIST images as unit 21, against the 21_1_2 MLP's test accuracy of 0.869.